# 循环神经网络（RNN / LSTM）

把每一行 `28` 个像素当作一步，共 `28` 步；使用 `nn.LSTM`，取最后一步隐状态接线性层分类（与仓库 `rnn.py` 一致）。


In [1]:
from pathlib import Path

import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms

torch.manual_seed(1)


## 超参数


In [2]:
EPOCH = 1
BATCH_SIZE = 64
TIME_STEP = 28
INPUT_SIZE = 28
LR = 0.01

TEST_N = 2000


## 数据


In [3]:
import numpy as np
from pathlib import Path

import torchvision
from mnist_from_raw import MNISTNumpyDataset, load_all_numpy, raw_files_available

if raw_files_available():
    train_x, train_y, te_imgs, te_lbls = load_all_numpy()
    train_data = MNISTNumpyDataset(train_x, train_y)
    train_loader = torch.utils.data.DataLoader(
        dataset=train_data, batch_size=BATCH_SIZE, shuffle=True
    )
    test_x = torch.from_numpy(np.ascontiguousarray(te_imgs[:TEST_N])).float().div_(255.0)
    test_y = te_lbls[:TEST_N]
    print("数据来源: data/raw")
else:
    MNIST_ROOT = Path("./mnist")
    download = not MNIST_ROOT.is_dir() or not any(MNIST_ROOT.iterdir())
    train_data = dsets.MNIST(
        root=str(MNIST_ROOT),
        train=True,
        transform=transforms.ToTensor(),
        download=download,
    )
    train_loader = torch.utils.data.DataLoader(
        dataset=train_data, batch_size=BATCH_SIZE, shuffle=True
    )
    test_data = dsets.MNIST(
        root=str(MNIST_ROOT), train=False, transform=transforms.ToTensor(), download=download
    )
    test_x = test_data.test_data.type(torch.FloatTensor)[:TEST_N] / 255.0
    test_y = test_data.test_labels.numpy()[:TEST_N]
    print("数据来源: torchvision ->", MNIST_ROOT.resolve())
print("test_x:", test_x.shape)


数据来源: data/raw
test_x: torch.Size([2000, 28, 28])


/tmp/ipykernel_3281946/2537232591.py:13: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  test_x = torch.from_numpy(np.ascontiguousarray(te_imgs[:TEST_N])).float().div_(255.0)


## 模型：LSTM + 线性层


In [4]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size=INPUT_SIZE,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
        )
        self.out = nn.Linear(64, 10)

    def forward(self, x):
        r_out, _ = self.rnn(x, None)
        return self.out(r_out[:, -1, :])


model = RNN()
print(model)


RNN(
  (rnn): LSTM(28, 64, batch_first=True)
  (out): Linear(in_features=64, out_features=10, bias=True)
)


## 优化器与损失


In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()


## 训练

每个 batch：`(B, 1, 28, 28)` → `view` 成 `(B, 28, 28)` 作为 `(batch, time, input_size)`。


In [6]:
def accuracy(logits, y_tensor):
    y_tensor = torch.as_tensor(y_tensor, device=logits.device)
    return (logits.argmax(dim=1) == y_tensor).float().mean().item()

for epoch in range(EPOCH):
    for step, (b_x, b_y) in enumerate(train_loader):
        b_x = b_x.view(-1, TIME_STEP, INPUT_SIZE)
        logits = model(b_x)
        loss = loss_fn(logits, b_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            with torch.no_grad():
                te = model(test_x)
                acc = accuracy(te, test_y)
            print(f"epoch={epoch} step={step} loss={loss.item():.4f} test_acc={acc:.4f}")


/data1/zdguo/document-parsing/alextools/experiment/MNIST/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


epoch=0 step=0 loss=2.3088 test_acc=0.0890
epoch=0 step=50 loss=1.3011 test_acc=0.5845
epoch=0 step=100 loss=0.7457 test_acc=0.7205
epoch=0 step=150 loss=0.6352 test_acc=0.7760
epoch=0 step=200 loss=0.2484 test_acc=0.8720
epoch=0 step=250 loss=0.2974 test_acc=0.8595
epoch=0 step=300 loss=0.1336 test_acc=0.9115
epoch=0 step=350 loss=0.4928 test_acc=0.9140
epoch=0 step=400 loss=0.1491 test_acc=0.9210
epoch=0 step=450 loss=0.1249 test_acc=0.9315
epoch=0 step=500 loss=0.1085 test_acc=0.9380
epoch=0 step=550 loss=0.2188 test_acc=0.9455
epoch=0 step=600 loss=0.1688 test_acc=0.9515
epoch=0 step=650 loss=0.1961 test_acc=0.9475
epoch=0 step=700 loss=0.0330 test_acc=0.9510
epoch=0 step=750 loss=0.2744 test_acc=0.9365
epoch=0 step=800 loss=0.0944 test_acc=0.9445
epoch=0 step=850 loss=0.4130 test_acc=0.9535
epoch=0 step=900 loss=0.2915 test_acc=0.9440


## 预测样例


In [7]:
model.eval()
with torch.no_grad():
    pred = model(test_x[:10].view(-1, TIME_STEP, INPUT_SIZE)).argmax(dim=1).cpu().numpy()
print("pred:", pred)
print("true:", test_y[:10])


pred: [7 2 1 0 4 1 4 9 5 9]
true: [7 2 1 0 4 1 4 9 5 9]
